In [69]:
from importlib import import_module
from importlib.metadata import version, PackageNotFoundError, distributions
import json

# utils_list_versions.py
# Run this cell to list installed versions for langchain / langgraph related packages and modules.


# Candidate distribution names and module names to probe
candidate_dists = [
    "langchain",
    "langgraph",
    "langchain-openai",
    "langchain_openai",
    "langchain-core",
    "langchain_core",
    "langchain-community",
    "langchain_community",
    "langgraph-prebuilt",
    "langgraph_prebuilt",
]
candidate_modules = [
    "langchain",
    "langgraph",
    "langchain_openai",
    "langchain_core",
    "langchain_community",
    "langgraph.prebuilt",
    "langchain_community.agent_toolkits",
    "langchain_community.tools.yahoo_finance_news",
    "langchain_community.agent_toolkits.zapier.toolkit",
]

def get_dist_versions(names):
    out = {}
    for name in names:
        try:
            out[name] = version(name)
        except PackageNotFoundError:
            out[name] = None
        except Exception as e:
            out[name] = f"error: {e}"
    return out

def get_module_versions(module_names):
    out = {}
    for mod_name in module_names:
        try:
            mod = import_module(mod_name)
            ver = getattr(mod, "__version__", None)
            # some packages expose version in __version__ within submodule __init__
            out[mod_name] = ver
        except Exception as e:
            out[mod_name] = None
    return out

def scan_installed_prefix(prefixes=("langchain", "langgraph")):
    found = {}
    for dist in distributions():
        name = dist.metadata["Name"] if dist.metadata else None
        if not name:
            continue
        lower = name.lower()
        if any(lower.startswith(p) for p in prefixes):
            try:
                found[name] = dist.version
            except Exception:
                found[name] = None
    return found

if __name__ == "__main__":
    dist_versions = get_dist_versions(candidate_dists)
    module_versions = get_module_versions(candidate_modules)
    scanned = scan_installed_prefix(("langchain", "langgraph", "langchain_community", "langchain-community"))

    result = {
        "probed_distributions": dist_versions,
        "probed_modules": module_versions,
        "scanned_installed_matches": scanned,
    }

    # Print readable JSON summary
    print(json.dumps(result, indent=2))

{
  "probed_distributions": {
    "langchain": "0.3.27",
    "langgraph": "0.6.7",
    "langchain-openai": "0.2.14",
    "langchain_openai": "0.2.14",
    "langchain-core": "0.3.76",
    "langchain_core": "0.3.76",
    "langchain-community": "0.3.29",
    "langchain_community": "0.3.29",
    "langgraph-prebuilt": "0.6.4",
    "langgraph_prebuilt": "0.6.4"
  },
  "probed_modules": {
    "langchain": "0.3.27",
    "langgraph": null,
    "langchain_openai": null,
    "langchain_core": "0.3.76",
    "langchain_community": "0.3.29",
    "langgraph.prebuilt": null,
    "langchain_community.agent_toolkits": null,
    "langchain_community.tools.yahoo_finance_news": null,
    "langchain_community.agent_toolkits.zapier.toolkit": null
  },
  "scanned_installed_matches": {
    "langgraph": "0.6.7",
    "langchain-community": "0.3.29",
    "langchain": "0.3.27",
    "langchain-openai": "0.2.14",
    "langchain-core": "0.3.76",
    "langchain-anthropic": "0.3.20",
    "langgraph-checkpoint": "2.1.1"

In [4]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

https://mcp.zapier.com/mcp/servers

# Reactive Agents

#### Build a Reactive Agent - how to analyze the market 

In [14]:
#!pip install yfinance

## Flow 
        User Query
        ↓
        LLM decides → which tool?
        ↓
        Tool executes
        ↓
        LLM observes result
        ↓
        Repeat...
        ↓
        Final Answer

#### Tools used 
        get_market_data
        get_market_news
        analyze_market

In [64]:
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
import yfinance as yf
import json
import re

# -------------------------
# LLM
# -------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# -------------------------
# Utility
# -------------------------
def clean_json(text: str) -> dict:
    cleaned = re.sub(r"```json|```", "", text).strip()
    return json.loads(cleaned)

# -------------------------
# TOOLS
# -------------------------

@tool
def get_market_data(query: str) -> dict:
    """Get recent market data using a query (NIFTY, TCS, etc.)."""

    prompt = f"""
    Extract the correct ticker from the query.

    Examples:
    NIFTY 50 -> ^NSEI
    SENSEX -> ^BSESN
    Reliance -> RELIANCE.NS
    TCS -> TCS.NS

    Query: {query}

    Respond ONLY in JSON:
    {{
      "ticker": "<valid ticker>"
    }}
    """

    response = llm.invoke(prompt)
    ticker = clean_json(response.content)["ticker"]

    data = yf.Ticker(ticker).history(period="5d")

    return {
        "ticker": ticker,
        "recent_data": data.tail().to_dict()
    }


@tool
def get_market_news(query: str) -> dict:
    """Get latest market news (LLM-generated)."""

    prompt = f"""
    Provide latest financial news for:

    {query}

    Respond ONLY in JSON:
    {{
      "news": [
        {{
          "headline": "...",
          "summary": "..."
        }}
      ]
    }}
    """

    response = llm.invoke(prompt)
    return clean_json(response.content)


@tool
def analyze_market(query: str) -> dict:
    """Analyze market trend."""

    prompt = f"""
    Analyze market trend for:

    {query}

    Respond ONLY in JSON:
    {{
      "trend": "<Bullish / Bearish / Sideways>",
      "reason": "...",
      "risk_level": "<Low / Medium / High>"
    }}
    """

    response = llm.invoke(prompt)
    return clean_json(response.content)


# -------------------------
# TOOLS
# -------------------------
tools = [get_market_data, get_market_news, analyze_market]

# -------------------------
# PROMPT
# -------------------------
prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a finance assistant.

Rules:
- Use get_market_data for prices
- Use get_market_news for news
- Use analyze_market for reasoning
- Combine outputs before final answer
"""),
    MessagesPlaceholder(variable_name="messages"),
])

# -------------------------
# AGENT (LangGraph)
# -------------------------
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=prompt
)

# -------------------------
# RUN (NO AgentExecutor)
# -------------------------
result = agent.invoke({
    "messages": [
        HumanMessage(content="Analyze NIFTY 50 and provide trend and latest news")
    ]
})

print(result)

{'messages': [HumanMessage(content='Analyze NIFTY 50 and provide trend and latest news', additional_kwargs={}, response_metadata={}, id='513627e1-46ba-47ed-bc26-477fd6568be8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_8XQmQtNpk2v1TptNqpR1Qysk', 'function': {'arguments': '{"query": "NIFTY 50"}', 'name': 'get_market_data'}, 'type': 'function'}, {'id': 'call_NfgIZMw8Cu3x45bm5fxKJcJn', 'function': {'arguments': '{"query": "NIFTY 50"}', 'name': 'get_market_news'}, 'type': 'function'}, {'id': 'call_dzlDxwMg6JQDQALjjxiqcqTV', 'function': {'arguments': '{"query": "NIFTY 50"}', 'name': 'analyze_market'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 148, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4

In [65]:
from IPython.display import Markdown, display
import json

In [66]:
from IPython.display import Markdown, display
import json

# Run agent
result = agent.invoke({
    "messages": [
        HumanMessage(content="Analyze NIFTY 50 and provide trend and latest news")
    ]
})

# -------------------------
# Extract Final Output
# -------------------------
# LangGraph returns messages list → last message is final answer
final_message = result["messages"][-1].content

# -------------------------
# Display nicely
# -------------------------
display(Markdown(f"### 📊 Finance Analysis\n\n{final_message}"))

### 📊 Finance Analysis

### NIFTY 50 Market Analysis

#### Recent Market Data
- **Open Prices**: 
  - March 16: 23,116.10
  - March 17: 23,493.20
  - March 18: 23,632.90
  - March 19: 23,197.75
  - March 20: 23,110.15
- **Close Prices**: 
  - March 16: 23,408.80
  - March 17: 23,581.15
  - March 18: 23,777.80
  - March 19: 23,002.15
  - March 20: 23,114.50
- **High Prices**: 
  - March 16: 23,502.00
  - March 17: 23,656.80
  - March 18: 23,862.25
  - March 19: 23,378.70
  - March 20: 23,345.15
- **Low Prices**: 
  - March 16: 22,955.25
  - March 17: 23,346.60
  - March 18: 23,618.45
  - March 19: 22,930.35
  - March 20: 23,067.60
- **Volume**: 
  - March 16: 540,300
  - March 17: 458,800
  - March 18: 382,900
  - March 19: 550,100
  - March 20: Data not available

#### Market Trend
- **Trend**: Bullish
- **Reason**: Recent economic indicators show strong GDP growth, positive corporate earnings reports, and increased foreign investment, contributing to upward momentum in the NIFTY 50 index.
- **Risk Level**: Medium

#### Latest News
1. **NIFTY 50 Hits Record High Amid Positive Global Cues**: The NIFTY 50 index reached a new all-time high today, driven by strong buying interest in technology and financial stocks, as global markets rallied on optimism over economic recovery.
   
2. **NIFTY 50 Closes Higher as Investors Cheer Earnings Reports**: The NIFTY 50 closed up 1.2% following the release of better-than-expected quarterly earnings from major companies, boosting investor sentiment and market confidence.
   
3. **NIFTY 50 Volatility Increases Ahead of Key Economic Data Release**: Market analysts are observing increased volatility in the NIFTY 50 as investors await crucial economic data that could influence future monetary policy decisions.

### Summary
The NIFTY 50 is currently in a bullish trend, supported by strong economic indicators and positive corporate earnings. Recent news highlights a record high for the index and increased investor confidence, although there is some volatility as key economic data is anticipated.

# Build Portfolio

In [67]:
portfolio = [
    {"stock": "RELIANCE.NS", "sector": "Energy", "purchase_price": 2400, "quantity": 50, "value": 125000},
    {"stock": "TCS.NS", "sector": "IT", "purchase_price": 3200, "quantity": 30, "value": 120000},
    {"stock": "HDFCBANK.NS", "sector": "Banking", "purchase_price": 1600, "quantity": 60, "value": 120000},
    {"stock": "ICICIBANK.NS", "sector": "Banking", "purchase_price": 950, "quantity": 100, "value": 110000},
    {"stock": "INFY.NS", "sector": "IT", "purchase_price": 1450, "quantity": 70, "value": 105000},
    {"stock": "LT.NS", "sector": "Infra", "purchase_price": 2200, "quantity": 40, "value": 100000},
    {"stock": "BHARTIARTL.NS", "sector": "Telecom", "purchase_price": 850, "quantity": 110, "value": 95000},
    {"stock": "ITC.NS", "sector": "FMCG", "purchase_price": 400, "quantity": 200, "value": 90000},
    {"stock": "HINDUNILVR.NS", "sector": "FMCG", "purchase_price": 2500, "quantity": 30, "value": 90000},
    {"stock": "SBIN.NS", "sector": "Banking", "purchase_price": 600, "quantity": 140, "value": 85000},
    {"stock": "AXISBANK.NS", "sector": "Banking", "purchase_price": 900, "quantity": 90, "value": 80000},
    {"stock": "TATAMOTORS.NS", "sector": "Auto", "purchase_price": 500, "quantity": 150, "value": 75000},
    {"stock": "SUNPHARMA.NS", "sector": "Pharma", "purchase_price": 1000, "quantity": 70, "value": 70000},
    {"stock": "ASIANPAINT.NS", "sector": "Paints", "purchase_price": 3000, "quantity": 20, "value": 65000},
    {"stock": "ADANIGREEN.NS", "sector": "Energy", "purchase_price": 1200, "quantity": 40, "value": 70000}
]

In [68]:
import pandas as pd
df = pd.DataFrame(portfolio)
df.head()

,stock,sector,purchase_price,quantity,value
0,RELIANCE.NS,Energy,2400,50,125000
1,TCS.NS,IT,3200,30,120000
2,HDFCBANK.NS,Banking,1600,60,120000
3,ICICIBANK.NS,Banking,950,100,110000
4,INFY.NS,IT,1450,70,105000


        Build Agent which can take action on Portfolio 
        This is a Reactive Agent who takes decision on a Portfolio
        This uses Langgraph based approach , State captures the internal state
        There is a planner Node that decides what steps have to be taken 
        Pydantic enforces the structure
        Decision Node takes decision on Buy Sell or Hold 
            - It has 2 variant one with reason other without Reason

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
import pandas as pd

# -------------------------
# LLM
# -------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------
# STATE (Pydantic)
# -------------------------
class AgentState(BaseModel):
    query: str
    plan: dict | None = None
    news: dict | None = None
    decision: dict | None = None


# -------------------------
# PORTFOLIO
# -------------------------
# portfolio = [
#     {"stock": "RELIANCE.NS", "sector": "Energy", "value": 125000},
#     {"stock": "TCS.NS", "sector": "IT", "value": 120000},
#     {"stock": "HDFCBANK.NS", "sector": "Banking", "value": 120000},
#     {"stock": "ICICIBANK.NS", "sector": "Banking", "value": 110000},
#     {"stock": "INFY.NS", "sector": "IT", "value": 105000},
# ]

df = pd.DataFrame(portfolio)

# -------------------------
# OUTPUT SCHEMAS
# -------------------------
class PlanOutput(BaseModel):
    steps: list[str]

class NewsItem(BaseModel):
    headline: str
    impact: str

class NewsOutput(BaseModel):
    news: list[NewsItem]

class DecisionItem(BaseModel):
    stock: str
    action: str
    reason: str

class DecisionOutput(BaseModel):
    actions: list[DecisionItem]


# with_structured_output -- It forces the LLM to return output in a strict schema (Pydantic model),instead of free-form text.
# -------------------------
# NODE 1: PLANNER
# -------------------------
def planner_node(state: AgentState):

    structured_llm = llm.with_structured_output(PlanOutput)

    plan = structured_llm.invoke(f"""
    Create a plan for:

    {state.query}
    """)

    return state.model_copy(update={"plan": plan.dict()})

# state.model_copy
# Take old state
# + apply changes
# → return NEW state object

# -------------------------
# NODE 2: NEWS
# -------------------------
def news_node(state: AgentState):

    structured_llm = llm.with_structured_output(NewsOutput)

    news = structured_llm.invoke(f"""
    Provide latest financial news impacting:

    {state.query}
    """)

    return state.model_copy(update={"news": news.dict()})


# -------------------------
# NODE 3: DECISION
# -------------------------
def decision_node(state: AgentState):

    structured_llm = llm.with_structured_output(DecisionOutput)
# Variant 1 -- No Reasoning asked but Pydantic Forces LLM to get the reason
# Issue - can create Hallucinations

    # decision = structured_llm.invoke(f"""
    # Based on this portfolio and news, suggest actions.

    # PORTFOLIO:
    # {df.to_dict(orient="records")}

    # NEWS:
    # {state.news}
    # """)
# Variant 2 -- Seeking Reason 
    decision = structured_llm.invoke(f"""
    Based on this portfolio and news, suggest actions.

    For EACH stock:
    - Decide BUY / SELL / HOLD
    - Provide a clear reason based on:
        - News impact
        - Sector exposure
        - Portfolio balance

    PORTFOLIO:
    {df.to_dict(orient="records")}

    NEWS:
    {state.news}
    """)

    return state.model_copy(update={"decision": decision.dict()})


# -------------------------
# GRAPH
# -------------------------
builder = StateGraph(AgentState)

builder.add_node("planner", planner_node)
builder.add_node("news", news_node)
builder.add_node("decision", decision_node)

builder.set_entry_point("planner")
builder.add_edge("planner", "news")
builder.add_edge("news", "decision")
builder.add_edge("decision", END)

graph = builder.compile()

# -------------------------
# RUN
# -------------------------
result = graph.invoke(
    AgentState(
        query="Analyze my portfolio and suggest actions based on latest market news"
    )
)



/var/folders/m8/lr5c6v793qbcszm54dsf6y440000gn/T/ipykernel_23660/1746880013.py:71: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return state.model_copy(update={"plan": plan.dict()})
/var/folders/m8/lr5c6v793qbcszm54dsf6y440000gn/T/ipykernel_23660/1746880013.py:87: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return state.model_copy(update={"news": news.dict()})
/var/folders/m8/lr5c6v793qbcszm54dsf6y440000gn/T/ipykernel_23660/1746880013.py:126: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/m

In [49]:
# -------------------------
# DISPLAY
# -------------------------
from IPython.display import Markdown, display

display(Markdown(f"""
## 📊 Proactive Portfolio Agent

### 🧭 Plan
{result["plan"]}

### 📰 News
{result["news"]}

### 📈 Decision
{result["decision"]}
"""))


## 📊 Proactive Portfolio Agent

### 🧭 Plan
{'steps': ['Gather current market news and trends from reliable financial news sources.', 'Review the current performance of your portfolio, including asset allocation and individual asset performance.', 'Identify any assets in your portfolio that are underperforming or overperforming based on the latest market news.', 'Assess the impact of recent market news on the sectors and industries represented in your portfolio.', 'Consider potential actions such as rebalancing your portfolio, selling underperforming assets, or investing in new opportunities that align with market trends.', 'Create a timeline for implementing any changes to your portfolio based on your analysis.', 'Monitor the market and your portfolio regularly to adjust your strategy as needed.']}

### 📰 News
{'news': [{'headline': 'Federal Reserve Signals Interest Rate Hike May Be Delayed', 'impact': 'Positive for growth stocks, negative for bonds.'}, {'headline': 'Tech Stocks Surge as Earnings Reports Exceed Expectations', 'impact': 'Positive for technology sector investments.'}, {'headline': 'Oil Prices Drop Amid Global Supply Concerns', 'impact': 'Negative for energy sector investments.'}, {'headline': 'Inflation Rates Show Signs of Stabilization', 'impact': 'Positive for consumer spending and retail stocks.'}, {'headline': 'Geopolitical Tensions Rise, Affecting Market Sentiment', 'impact': 'Negative for overall market stability.'}]}

### 📈 Decision
{'actions': [{'stock': 'RELIANCE.NS', 'action': 'SELL', 'reason': "Oil prices are dropping, negatively impacting the energy sector, and Reliance's performance may be affected."}, {'stock': 'TCS.NS', 'action': 'BUY', 'reason': 'Tech stocks are surging due to positive earnings reports, making TCS a strong buy in the IT sector.'}, {'stock': 'HDFCBANK.NS', 'action': 'HOLD', 'reason': 'Stable banking sector performance; no immediate news impact suggests holding is prudent.'}, {'stock': 'ICICIBANK.NS', 'action': 'HOLD', 'reason': 'Similar to HDFC Bank, ICICI Bank is stable; holding is advisable amidst geopolitical tensions.'}, {'stock': 'INFY.NS', 'action': 'BUY', 'reason': 'Positive earnings reports in the tech sector indicate a good opportunity to buy Infosys.'}, {'stock': 'LT.NS', 'action': 'HOLD', 'reason': 'Infrastructure sector remains stable; no immediate news suggests a need to sell.'}, {'stock': 'BHARTIARTL.NS', 'action': 'HOLD', 'reason': 'Telecom sector is stable; holding is advisable amidst market uncertainties.'}, {'stock': 'ITC.NS', 'action': 'BUY', 'reason': 'Inflation stabilization is positive for FMCG stocks, making ITC a good buy.'}, {'stock': 'HINDUNILVR.NS', 'action': 'BUY', 'reason': 'Similar to ITC, Hindustan Unilever benefits from stabilized inflation, making it a strong buy.'}, {'stock': 'SBIN.NS', 'action': 'HOLD', 'reason': 'Stable banking sector; holding is advisable amidst geopolitical tensions.'}, {'stock': 'AXISBANK.NS', 'action': 'HOLD', 'reason': 'Stable performance in the banking sector; no immediate need to sell.'}, {'stock': 'TATAMOTORS.NS', 'action': 'HOLD', 'reason': 'Auto sector stability; holding is advisable amidst market uncertainties.'}, {'stock': 'SUNPHARMA.NS', 'action': 'HOLD', 'reason': 'Pharma sector stability; holding is advisable amidst market uncertainties.'}, {'stock': 'ASIANPAINT.NS', 'action': 'HOLD', 'reason': 'Paints sector stability; holding is advisable amidst market uncertainties.'}, {'stock': 'ADANIGREEN.NS', 'action': 'SELL', 'reason': 'Negative impact from dropping oil prices on energy sector investments.'}]}


In [50]:
result["plan"]

{'steps': ['Gather current market news and trends from reliable financial news sources.',
  'Review the current performance of your portfolio, including asset allocation and individual asset performance.',
  'Identify any assets in your portfolio that are underperforming or overperforming based on the latest market news.',
  'Assess the impact of recent market news on the sectors and industries represented in your portfolio.',
  'Consider potential actions such as rebalancing your portfolio, selling underperforming assets, or investing in new opportunities that align with market trends.',
  'Create a timeline for implementing any changes to your portfolio based on your analysis.',
  'Monitor the market and your portfolio regularly to adjust your strategy as needed.']}

In [51]:
result["decision"]

{'actions': [{'stock': 'RELIANCE.NS',
   'action': 'SELL',
   'reason': "Oil prices are dropping, negatively impacting the energy sector, and Reliance's performance may be affected."},
  {'stock': 'TCS.NS',
   'action': 'BUY',
   'reason': 'Tech stocks are surging due to positive earnings reports, making TCS a strong buy in the IT sector.'},
  {'stock': 'HDFCBANK.NS',
   'action': 'HOLD',
   'reason': 'Stable banking sector performance; no immediate news impact suggests holding is prudent.'},
  {'stock': 'ICICIBANK.NS',
   'action': 'HOLD',
   'reason': 'Similar to HDFC Bank, ICICI Bank is stable; holding is advisable amidst geopolitical tensions.'},
  {'stock': 'INFY.NS',
   'action': 'BUY',
   'reason': 'Positive earnings reports in the tech sector indicate a good opportunity to buy Infosys.'},
  {'stock': 'LT.NS',
   'action': 'HOLD',
   'reason': 'Infrastructure sector remains stable; no immediate news suggests a need to sell.'},
  {'stock': 'BHARTIARTL.NS',
   'action': 'HOLD',
  

# Full Reasoning Based 

        Planner → News → Decision → Validation → (Correction Loop) → END

        Upgrading the ReactAgent to have a reasoning
        Adding Validation node as a Feedback Loop, this node will decide if decision taken is correct, if not what can be done to correct it
        Validation captures score, out of 10 (using LLM) - based on score correction will be done
        AgentState(BaseModel) -- new state as validation is updated

        Design Changes
            1. Adding Validation Node as LLM-as-Judge will act as Feedback Node [Evaluate ALL these decisions together]
            2. Scoring Mechanism will make decision quantitative
            3. Error Handling handled using validation node
            4. Reasoning depth increased using FewShotLearning
            5. Decision Reliability increased using evaluation (via Validation)

In [58]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import pandas as pd

# -------------------------
# LLM
# -------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------
# STATE
# -------------------------
class AgentState(BaseModel):
    query: str
    plan: dict | None = None
    news: dict | None = None
    decision: dict | None = None
    validation: dict | None = None


# -------------------------
# PORTFOLIO
# -------------------------
portfolio = [
    {"stock": "RELIANCE.NS", "sector": "Energy", "value": 125000},
    {"stock": "TCS.NS", "sector": "IT", "value": 120000},
    {"stock": "HDFCBANK.NS", "sector": "Banking", "value": 120000},
    {"stock": "ICICIBANK.NS", "sector": "Banking", "value": 110000},
]

df = pd.DataFrame(portfolio)

# -------------------------
# OUTPUT SCHEMAS
# -------------------------
class PlanOutput(BaseModel):
    steps: list[str]

class NewsOutput(BaseModel):
    news: list

class DecisionItem(BaseModel):
    stock: str
    action: str
    reason: str

class DecisionOutput(BaseModel):
    actions: list[DecisionItem]

class ValidationOutput(BaseModel):
    score: int
    is_valid: bool
    issues: str
    confidence: str


# -------------------------
# NODE 1: PLANNER
# -------------------------
def planner_node(state: AgentState):
    structured_llm = llm.with_structured_output(PlanOutput)

    plan = structured_llm.invoke(f"""
    Create a step-by-step plan for:

    {state.query}
    """)

    return state.model_copy(update={"plan": plan.dict()})


# -------------------------
# NODE 2: NEWS
# -------------------------
def news_node(state: AgentState):
    structured_llm = llm.with_structured_output(NewsOutput)

    news = structured_llm.invoke(f"""
    Provide latest financial news impacting:

    {state.query}
    """)

    return state.model_copy(update={"news": news.dict()})


# -------------------------
# NODE 3: DECISION
# -------------------------
def decision_node(state: AgentState):
    structured_llm = llm.with_structured_output(DecisionOutput)

    decision = structured_llm.invoke(f"""
    Based on this portfolio and news, suggest actions.

    For EACH stock:
    - Decide BUY / SELL / HOLD
    - Provide reason using news + portfolio context

    PORTFOLIO:
    {df.to_dict(orient="records")}

    NEWS:
    {state.news}
    """)

    return state.model_copy(update={"decision": decision.dict()})


# -------------------------
# NODE 4: VALIDATION (LLM JUDGE)
# -------------------------
def validation_node(state: AgentState):

    structured_llm = llm.with_structured_output(ValidationOutput)

    validation = structured_llm.invoke(f"""
    You are an expert financial auditor.

    Evaluate the following investment decisions.

    ### Few-shot Examples:

    Example 1:
    Decision: SELL bank stocks due to rising NPAs
    News: Reports show increasing NPAs in banking sector
    → Good decision

    Example 2:
    Decision: BUY IT stocks
    News: IT sector facing slowdown
    → Bad decision

    ---

    Now evaluate:

    DECISION:
    {state.decision}

    NEWS:
    {state.news}

    Score from 1–10 based on:
    - Alignment with news
    - Logical reasoning
    - Portfolio diversification
    - Risk awareness

    Respond with:
    - score
    - is_valid (true if score >= 7)
    - issues
    - confidence (High/Medium/Low)
    """)

    return state.model_copy(update={"validation": validation.dict()})


# -------------------------
# NODE 5: CORRECTION LOOP
# -------------------------
def correction_node(state: AgentState):

    if state.validation["score"] >= 7:
        return state  # no change

    structured_llm = llm.with_structured_output(DecisionOutput)

    corrected = structured_llm.invoke(f"""
        The previous decision was flawed.

        ISSUES:
        {state.validation["issues"]}

        Re-generate improved decisions.

        PORTFOLIO:
        {df.to_dict(orient="records")}

        NEWS:
        {state.news}
        """)

    return state.model_copy(update={"decision": corrected.dict()})


# -------------------------
# ROUTER (DECISION)
# -------------------------
def should_correct(state: AgentState):
    if state.validation["score"] < 7:
        return "correct"
    return "end"


# -------------------------
# GRAPH
# -------------------------
builder = StateGraph(AgentState)

builder.add_node("planner", planner_node)
builder.add_node("news", news_node)
builder.add_node("decision", decision_node)
builder.add_node("validation", validation_node)
builder.add_node("correction", correction_node)

builder.set_entry_point("planner")

builder.add_edge("planner", "news")
builder.add_edge("news", "decision")
builder.add_edge("decision", "validation")

builder.add_conditional_edges(
    "validation",
    should_correct,
    {
        "correct": "correction",
        "end": END
    }
)

builder.add_edge("correction", "validation")

graph = builder.compile()

# -------------------------
# RUN
# -------------------------
result = graph.invoke(
    AgentState(
        query="Analyze my portfolio and suggest actions based on latest market news"
    )
)



/var/folders/m8/lr5c6v793qbcszm54dsf6y440000gn/T/ipykernel_23660/2469484872.py:70: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return state.model_copy(update={"plan": plan.dict()})
/var/folders/m8/lr5c6v793qbcszm54dsf6y440000gn/T/ipykernel_23660/2469484872.py:85: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return state.model_copy(update={"news": news.dict()})
/var/folders/m8/lr5c6v793qbcszm54dsf6y440000gn/T/ipykernel_23660/2469484872.py:108: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/m

In [59]:
# -------------------------
# DISPLAY
# -------------------------
from IPython.display import Markdown, display

display(Markdown(f"""
## 🧠 Reasoning-Based Portfolio Agent

### 🧭 Plan
{result['plan']}

### 📰 News
{result['news']}

### 📈 Decision
{result['decision']}

### 🧠 Validation
{result['validation']}
"""))


## 🧠 Reasoning-Based Portfolio Agent

### 🧭 Plan
{'steps': ['Gather all relevant information about your current portfolio, including asset types, quantities, and current market values.', 'Research the latest market news and trends that may impact your portfolio, focusing on economic indicators, sector performance, and geopolitical events.', 'Analyze the performance of each asset in your portfolio in relation to the latest market news, identifying any that may be underperforming or overperforming.', 'Evaluate the overall diversification of your portfolio to ensure it aligns with your risk tolerance and investment goals.', 'Based on the analysis, suggest specific actions such as rebalancing the portfolio, selling underperforming assets, or investing in new opportunities that align with market trends.', 'Create a timeline for implementing the suggested actions, including any necessary research or consultations with financial advisors.', 'Monitor the portfolio regularly and stay updated on market news to make informed decisions in the future.']}

### 📰 News
{'news': [{'headline': 'Federal Reserve Signals Interest Rate Hikes May Be Paused', 'summary': 'The Federal Reserve indicated that it may pause interest rate hikes in the coming months, which could lead to a more favorable environment for equities.'}, {'headline': 'Tech Stocks Rally as Earnings Exceed Expectations', 'summary': "Major tech companies reported earnings that surpassed analysts' expectations, leading to a rally in tech stocks."}, {'headline': 'Inflation Rates Show Signs of Easing', 'summary': 'Recent data shows that inflation rates are beginning to ease, which could impact consumer spending and economic growth.'}, {'headline': 'Oil Prices Surge Amid Geopolitical Tensions', 'summary': 'Geopolitical tensions have led to a surge in oil prices, impacting energy stocks and inflation concerns.'}, {'headline': 'Consumer Confidence Hits Highest Level in Two Years', 'summary': 'Consumer confidence has reached its highest level in two years, suggesting potential growth in consumer spending.'}]}

### 📈 Decision
{'actions': [{'stock': 'RELIANCE.NS', 'action': 'SELL', 'reason': "Oil prices have surged due to geopolitical tensions, which could negatively impact energy stocks like Reliance. It's prudent to take profits before potential declines."}, {'stock': 'TCS.NS', 'action': 'BUY', 'reason': 'The tech sector is rallying as major companies report earnings that exceed expectations. TCS, being a key player in IT, is likely to benefit from this trend.'}, {'stock': 'HDFCBANK.NS', 'action': 'HOLD', 'reason': 'With the Federal Reserve signaling a pause in interest rate hikes, banking stocks like HDFCBANK may stabilize. Holding is advisable to see how the market reacts.'}, {'stock': 'ICICIBANK.NS', 'action': 'HOLD', 'reason': 'Similar to HDFCBANK, ICICIBANK can benefit from a stable interest rate environment. Holding allows for potential gains as consumer confidence rises.'}]}

### 🧠 Validation
{'score': 8, 'is_valid': True, 'issues': '', 'confidence': 'High'}


In [60]:
result['decision']

{'actions': [{'stock': 'RELIANCE.NS',
   'action': 'SELL',
   'reason': "Oil prices have surged due to geopolitical tensions, which could negatively impact energy stocks like Reliance. It's prudent to take profits before potential declines."},
  {'stock': 'TCS.NS',
   'action': 'BUY',
   'reason': 'The tech sector is rallying as major companies report earnings that exceed expectations. TCS, being a key player in IT, is likely to benefit from this trend.'},
  {'stock': 'HDFCBANK.NS',
   'action': 'HOLD',
   'reason': 'With the Federal Reserve signaling a pause in interest rate hikes, banking stocks like HDFCBANK may stabilize. Holding is advisable to see how the market reacts.'},
  {'stock': 'ICICIBANK.NS',
   'action': 'HOLD',
   'reason': 'Similar to HDFCBANK, ICICIBANK can benefit from a stable interest rate environment. Holding allows for potential gains as consumer confidence rises.'}]}

In [61]:
result['validation']

{'score': 8, 'is_valid': True, 'issues': '', 'confidence': 'High'}

## In previous version --
    1. Correction could loop indefinitely (no control)
    2. Here putting cap to interation in case correction takes place
    3. Adding
        1. Decision History Tracking
        2. Decision Update Detection [Whether correction actually changed anything]
    4. Enhanced Observability
        1. Final decision
        2. Validation result
        3. Whether decision changed
        4. Number of iterations
        5. Full decision history


In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List, Optional
import pandas as pd

# -------------------------
# LLM
# -------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------
# STATE
# -------------------------
class AgentState(BaseModel):
    query: str
    plan: Optional[dict] = None
    news: Optional[dict] = None
    decision: Optional[dict] = None
    validation: Optional[dict] = None

    # Tracking
    decision_history: List[dict] = []
    decision_updated: bool = False
    iteration_count: int = 0


# -------------------------
# PORTFOLIO
# -------------------------
portfolio = [
    {"stock": "RELIANCE.NS", "sector": "Energy", "value": 125000},
    {"stock": "TCS.NS", "sector": "IT", "value": 120000},
    {"stock": "HDFCBANK.NS", "sector": "Banking", "value": 120000},
    {"stock": "ICICIBANK.NS", "sector": "Banking", "value": 110000},
]

df = pd.DataFrame(portfolio)

# -------------------------
# OUTPUT SCHEMAS
# -------------------------
class PlanOutput(BaseModel):
    steps: List[str]

class NewsItem(BaseModel):
    headline: str
    impact: str

class NewsOutput(BaseModel):
    news: List[NewsItem]

class DecisionItem(BaseModel):
    stock: str
    action: str
    reason: str

class DecisionOutput(BaseModel):
    actions: List[DecisionItem]

class ValidationOutput(BaseModel):
    score: int
    is_valid: bool
    issues: str
    confidence: str


# -------------------------
# NODE 1: PLANNER
# -------------------------
def planner_node(state: AgentState):
    structured_llm = llm.with_structured_output(PlanOutput)

    plan = structured_llm.invoke(f"""
    Create a step-by-step plan for:

    {state.query}
    """)

    return state.model_copy(update={"plan": plan.model_dump()})


# -------------------------
# NODE 2: NEWS
# -------------------------
def news_node(state: AgentState):
    structured_llm = llm.with_structured_output(NewsOutput)

    news = structured_llm.invoke(f"""
    Provide latest financial news impacting:

    {state.query}
    """)

    return state.model_copy(update={"news": news.model_dump()})


# -------------------------
# NODE 3: DECISION
# -------------------------
def decision_node(state: AgentState):
    structured_llm = llm.with_structured_output(DecisionOutput)

    decision = structured_llm.invoke(f"""
    Based on this portfolio and news, suggest actions.

    For EACH stock:
    - Decide BUY / SELL / HOLD
    - Provide reasoning using:
        - News impact
        - Sector exposure
        - Portfolio balance

    PORTFOLIO:
    {df.to_dict(orient="records")}

    NEWS:
    {state.news}
    """)

    decision_dict = decision.model_dump()

    return state.model_copy(update={
        "decision": decision_dict,
        "decision_history": [decision_dict],
        "decision_updated": False,
        "iteration_count": 0
    })


# -------------------------
# NODE 4: VALIDATION (LLM JUDGE)
# -------------------------
def validation_node(state: AgentState):
    structured_llm = llm.with_structured_output(ValidationOutput)

    validation = structured_llm.invoke(f"""
        You are an expert financial auditor.

        ### Few-shot Examples:

        Example 1:
        Decision: SELL bank stocks due to rising NPAs
        News: Banking NPAs increasing
        → Good decision

        Example 2:
        Decision: BUY IT stocks
        News: IT slowdown
        → Bad decision

        ---

        Evaluate the following:

        DECISION:
        {state.decision}

        NEWS:
        {state.news}

        Score (1–10) based on:
        - Alignment with news
        - Logical reasoning
        - Risk awareness
        - Portfolio diversification

        Return:
        - score
        - is_valid (true if score >= 7)
        - issues
        - confidence
        """)

    return state.model_copy(update={"validation": validation.model_dump()})


# -------------------------
# NODE 5: CORRECTION
# -------------------------
def correction_node(state: AgentState):

    # Stop if already valid
    if state.validation["score"] >= 7:
        return state

    structured_llm = llm.with_structured_output(DecisionOutput)

    corrected = structured_llm.invoke(f"""
The previous decision had issues:

{state.validation["issues"]}

Improve the decision.

PORTFOLIO:
{df.to_dict(orient="records")}

NEWS:
{state.news}
""")

    corrected_dict = corrected.model_dump()

    is_updated = corrected_dict != state.decision

    return state.model_copy(update={
        "decision": corrected_dict,
        "decision_history": state.decision_history + [corrected_dict],
        "decision_updated": state.decision_updated or is_updated,
        "iteration_count": state.iteration_count + 1
    })


# -------------------------
# ROUTER
# -------------------------
def should_correct(state: AgentState):
    if state.validation["score"] < 7 and state.iteration_count < 3:
        return "correct"
    return "end"


# -------------------------
# GRAPH
# -------------------------
builder = StateGraph(AgentState)

builder.add_node("planner", planner_node)
builder.add_node("news", news_node)
builder.add_node("decision", decision_node)
builder.add_node("validation", validation_node)
builder.add_node("correction", correction_node)

builder.set_entry_point("planner")

builder.add_edge("planner", "news")
builder.add_edge("news", "decision")
builder.add_edge("decision", "validation")

builder.add_conditional_edges(
    "validation",
    should_correct,
    {
        "correct": "correction",
        "end": END
    }
)

builder.add_edge("correction", "validation")

graph = builder.compile()

# -------------------------
# RUN
# -------------------------
result = graph.invoke(
    AgentState(
        query="Analyze my portfolio and suggest actions based on latest market news"
    )
)

# -------------------------
# DISPLAY
# -------------------------
from IPython.display import Markdown, display

display(Markdown(f"""
## 🧠 Reasoning-Based Portfolio Agent

### 🧭 Plan
{result["plan"]}

### 📰 News
{result["news"]}

### 📈 Final Decision
{result["decision"]}

### 🧠 Validation
{result["validation"]}

### 🔄 Decision Updated?
{result["decision_updated"]}

### 🔁 Iterations
{result["iteration_count"]}

### 📜 Decision History
{result["decision_history"]}
"""))


## 🧠 Reasoning-Based Portfolio Agent

### 🧭 Plan
{'steps': ['Gather all relevant information about your current portfolio, including asset types, quantities, and current market values.', 'Research the latest market news and trends that may impact your portfolio, focusing on economic indicators, sector performance, and geopolitical events.', 'Analyze the performance of each asset in your portfolio in relation to the latest market news, identifying any that may be underperforming or at risk.', 'Evaluate the overall diversification of your portfolio to ensure it aligns with your risk tolerance and investment goals.', 'Based on the analysis, suggest specific actions such as rebalancing the portfolio, selling underperforming assets, or investing in new opportunities that align with market trends.', 'Create a timeline for implementing the suggested actions, including any necessary research or consultations with financial advisors.', 'Monitor the market and your portfolio regularly to adjust your strategy as needed based on ongoing news and performance.']}

### 📰 News
{'news': [{'headline': 'Federal Reserve Signals Interest Rate Hike May Be Delayed', 'impact': 'Positive for growth stocks, negative for financials.'}, {'headline': 'Tech Stocks Rally as Earnings Exceed Expectations', 'impact': 'Positive for tech sector, potential for portfolio reallocation.'}, {'headline': 'Oil Prices Surge Amid Geopolitical Tensions', 'impact': 'Negative for consumers, positive for energy stocks.'}, {'headline': 'Inflation Rates Show Signs of Easing', 'impact': 'Positive for overall market sentiment, may boost consumer spending.'}, {'headline': "China's Economic Growth Slows, Impacting Global Markets", 'impact': 'Negative for commodities, potential risk for emerging markets.'}]}

### 📈 Final Decision
{'actions': [{'stock': 'RELIANCE.NS', 'action': 'BUY', 'reason': "Oil prices are surging due to geopolitical tensions, which is positive for energy stocks like Reliance. This aligns with the portfolio's exposure to the energy sector."}, {'stock': 'TCS.NS', 'action': 'BUY', 'reason': 'The tech sector is rallying as earnings exceed expectations, indicating strong growth potential. This is a good opportunity to increase exposure to IT stocks.'}, {'stock': 'HDFCBANK.NS', 'action': 'SELL', 'reason': "The Federal Reserve's signal of a delayed interest rate hike is negative for financials, and the portfolio is already heavily weighted in banking stocks."}, {'stock': 'ICICIBANK.NS', 'action': 'SELL', 'reason': "Similar to HDFCBANK, the negative impact of the Fed's signals on financials suggests reducing exposure in this sector, especially since the portfolio is already balanced towards banking."}]}

### 🧠 Validation
{'score': 8, 'is_valid': True, 'issues': '', 'confidence': 'high'}

### 🔄 Decision Updated?
False

### 🔁 Iterations
0

### 📜 Decision History
[{'actions': [{'stock': 'RELIANCE.NS', 'action': 'BUY', 'reason': "Oil prices are surging due to geopolitical tensions, which is positive for energy stocks like Reliance. This aligns with the portfolio's exposure to the energy sector."}, {'stock': 'TCS.NS', 'action': 'BUY', 'reason': 'The tech sector is rallying as earnings exceed expectations, indicating strong growth potential. This is a good opportunity to increase exposure to IT stocks.'}, {'stock': 'HDFCBANK.NS', 'action': 'SELL', 'reason': "The Federal Reserve's signal of a delayed interest rate hike is negative for financials, and the portfolio is already heavily weighted in banking stocks."}, {'stock': 'ICICIBANK.NS', 'action': 'SELL', 'reason': "Similar to HDFCBANK, the negative impact of the Fed's signals on financials suggests reducing exposure in this sector, especially since the portfolio is already balanced towards banking."}]}]


## Scope of Improvements --
    1. No contradiction check
    2. No Risk Control
        - BUY Reliance -- but check if the portfolio %age gets impacted
    3. Theme specific news 
    4. No hard rules [Fundamental Rules]

        Planner
        ↓
        News
        ↓
        📊 Portfolio Analytics  ✅ NEW
        ↓
        ⚠️ Risk Engine          ✅ NEW
        ↓
        Decision
        ↓
        Validation
        ↓
        Correction Loop

        User Query
        ↓
        Planner (LLM)
        ↓
        News (LLM)
        ↓
        Analytics (Deterministic)
        ↓
        Risk Engine (Rules)
        ↓
        Decision (LLM reasoning)
        ↓
        Validation (LLM critic)
        ↓
        ├── If bad → Correction → back to Validation
        └── If good → END

## Build conditional Loop -- 
##### Bad Decision as - score < 7 AND iteration_count < 3
##### Good Decision as - score >= 7 OR iteration_count >= 3
##### After Validation It ends

### Iterative Improvement Loop
        validation → correction → validation

## Design Choices
        1. Separation of concerns
            Analytics → deterministic
            Risk → rule-based
            Decision → LLM
        2. Structured output
            llm.with_structured_output
        3. Decision history tracking
            This enables: auditability, explainability, Future scope on improvements
        4. Guardrails via validation
            verify → correct → retry
        5. Where This Can Break
            news_node -- For Fake News 
            Risk engine is too simple
            Correction lacks full context

In [56]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List, Optional
import pandas as pd

# -------------------------
# LLM
# -------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------
# STATE
# -------------------------
class AgentState(BaseModel):
    query: str
    plan: Optional[dict] = None
    news: Optional[dict] = None
    analytics: Optional[dict] = None
    risk: Optional[dict] = None
    decision: Optional[dict] = None
    validation: Optional[dict] = None

    decision_history: List[dict] = []
    decision_updated: bool = False
    iteration_count: int = 0


# -------------------------
# PORTFOLIO
# -------------------------
portfolio = [
    {"stock": "RELIANCE.NS", "sector": "Energy", "value": 125000},
    {"stock": "TCS.NS", "sector": "IT", "value": 120000},
    {"stock": "HDFCBANK.NS", "sector": "Banking", "value": 120000},
    {"stock": "ICICIBANK.NS", "sector": "Banking", "value": 110000},
]

df = pd.DataFrame(portfolio)

# -------------------------
# SCHEMAS
# -------------------------
class PlanOutput(BaseModel):
    steps: List[str]

class NewsOutput(BaseModel):
    news: list

class DecisionItem(BaseModel):
    stock: str
    action: str
    reason: str

class DecisionOutput(BaseModel):
    actions: List[DecisionItem]

class ValidationOutput(BaseModel):
    score: int
    is_valid: bool
    issues: str
    confidence: str


# -------------------------
# NODE 1: PLANNER
# -------------------------
def planner_node(state: AgentState):
    structured_llm = llm.with_structured_output(PlanOutput)

    plan = structured_llm.invoke(f"Create plan for: {state.query}")

    return state.model_copy(update={"plan": plan.model_dump()})


# -------------------------
# NODE 2: NEWS
# -------------------------
def news_node(state: AgentState):
    structured_llm = llm.with_structured_output(NewsOutput)

    news = structured_llm.invoke(f"Provide financial news for: {state.query}")

    return state.model_copy(update={"news": news.model_dump()})


# -------------------------
# NODE 3: PORTFOLIO ANALYTICS (NEW)
# -------------------------
def analytics_node(state: AgentState):

    total_value = df["value"].sum()

    sector_alloc = df.groupby("sector")["value"].sum().to_dict()

    sector_pct = {
        k: round(v / total_value * 100, 2)
        for k, v in sector_alloc.items()
    }

    top_stock = df.loc[df["value"].idxmax()].to_dict()

    analytics = {
        "total_value": total_value,
        "sector_allocation_pct": sector_pct,
        "top_stock": top_stock
    }

    return state.model_copy(update={"analytics": analytics})


# -------------------------
# NODE 4: RISK ENGINE (NEW)
# -------------------------
def risk_node(state: AgentState):

    sector_pct = state.analytics["sector_allocation_pct"]

    risks = []

    for sector, pct in sector_pct.items():
        if pct > 25:
            risks.append(f"{sector} overexposed ({pct}%)")

    if not risks:
        risks.append("No major concentration risk")

    risk_output = {
        "risks": risks
    }

    return state.model_copy(update={"risk": risk_output})


# -------------------------
# NODE 5: DECISION
# -------------------------
def decision_node(state: AgentState):
    structured_llm = llm.with_structured_output(DecisionOutput)

    decision = structured_llm.invoke(f"""
    Suggest BUY/SELL/HOLD.

    Consider:
    - News
    - Portfolio analytics
    - Risk signals

    ANALYTICS:
    {state.analytics}

    RISK:
    {state.risk}

    NEWS:
    {state.news}
    """)

    decision_dict = decision.model_dump()

    return state.model_copy(update={
        "decision": decision_dict,
        "decision_history": [decision_dict],
        "decision_updated": False,
        "iteration_count": 0
    })


# -------------------------
# NODE 6: VALIDATION
# -------------------------
def validation_node(state: AgentState):
    structured_llm = llm.with_structured_output(ValidationOutput)

    validation = structured_llm.invoke(f"""
    Evaluate decision quality.

    Must check:
    - Uses analytics?
    - Respects risk?
    - Matches news?

    DECISION:
    {state.decision}

    ANALYTICS:
    {state.analytics}

    RISK:
    {state.risk}

    NEWS:
    {state.news}
    """)

    return state.model_copy(update={"validation": validation.model_dump()})


# -------------------------
# NODE 7: CORRECTION
# -------------------------
def correction_node(state: AgentState):

    if state.validation["score"] >= 7:
        return state

    structured_llm = llm.with_structured_output(DecisionOutput)

    corrected = structured_llm.invoke(f"""
    Fix decision using:
    - analytics
    - risk signals

    ISSUES:
    {state.validation["issues"]}
    """)

    corrected_dict = corrected.model_dump()

    is_updated = corrected_dict != state.decision

    return state.model_copy(update={
        "decision": corrected_dict,
        "decision_history": state.decision_history + [corrected_dict],
        "decision_updated": state.decision_updated or is_updated,
        "iteration_count": state.iteration_count + 1
    })


# -------------------------
# ROUTER
# -------------------------
def should_correct(state: AgentState):
    if state.validation["score"] < 7 and state.iteration_count < 3:
        return "correct"
    return "end"


# -------------------------
# GRAPH
# -------------------------
builder = StateGraph(AgentState)

builder.add_node("planner", planner_node)
builder.add_node("news", news_node)
builder.add_node("analytics", analytics_node)
builder.add_node("risk", risk_node)
builder.add_node("decision", decision_node)
builder.add_node("validation", validation_node)
builder.add_node("correction", correction_node)

builder.set_entry_point("planner")

builder.add_edge("planner", "news")
builder.add_edge("news", "analytics")
builder.add_edge("analytics", "risk")
builder.add_edge("risk", "decision")
builder.add_edge("decision", "validation")

builder.add_conditional_edges(
    "validation",
    should_correct,
    {"correct": "correction", "end": END}
)

builder.add_edge("correction", "validation")

graph = builder.compile()

# -------------------------
# RUN
# -------------------------
result = graph.invoke(
    AgentState(query="Analyze portfolio and suggest actions")
)

# -------------------------
# DISPLAY
# -------------------------
from IPython.display import Markdown, display

display(Markdown(f"""
## 🚀 Hybrid Financial Agent

### 📊 Analytics
{result["analytics"]}

### ⚠️ Risk
{result["risk"]}

### 📈 Decision
{result["decision"]}

### 🧠 Validation
{result["validation"]}

### 🔄 Updated?
{result["decision_updated"]}

### 🔁 Iterations
{result["iteration_count"]}
"""))


## 🚀 Hybrid Financial Agent

### 📊 Analytics
{'total_value': 475000, 'sector_allocation_pct': {'Banking': 48.42, 'Energy': 26.32, 'IT': 25.26}, 'top_stock': {'stock': 'RELIANCE.NS', 'sector': 'Energy', 'value': 125000}}

### ⚠️ Risk
{'risks': ['Banking overexposed (48.42%)', 'Energy overexposed (26.32%)', 'IT overexposed (25.26%)']}

### 📈 Decision
{'actions': [{'stock': 'RELIANCE.NS', 'action': 'SELL', 'reason': 'Overexposure in Energy sector (26.32%) and recent market volatility.'}, {'stock': 'Banking Sector', 'action': 'SELL', 'reason': 'High exposure (48.42%) and potential risks from interest rate hikes.'}, {'stock': 'IT Sector', 'action': 'HOLD', 'reason': 'Current mixed performance; reassess after further analysis.'}]}

### 🧠 Validation
{'score': 85, 'is_valid': True, 'issues': '', 'confidence': 'high'}

### 🔄 Updated?
False

### 🔁 Iterations
0
